# Training Pipeline — LSH-Distilled Variant

Trains the global anomaly detector and the 17 fault-specialist XGBoost models on the **LSH-distilled** training set (~90% smaller than the full dataset — see `detector_lsh_final.ipynb` for the distillation step). Hyperparameters are tuned with Optuna; SHAP is used to select the top-10 features per specialist. Outputs `.joblib` model packages to `../models/lsh/`.

In [ ]:
import os, warnings, joblib, shap, optuna, time
import numpy as np
import pandas as pd
import pyreadr
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ==============================================================================
# NOTE: paths below are relative to this notebook living in /notebooks.
# Place the TEP dataset files in /data and trained model artifacts in /models
# (see the repository README for download links and folder layout).
# 0. CONFIGURAÇÕES E AUXILIARES
# ==============================================================================
PASTA_MODELOS = os.path.join("..", "models", "lsh")
CAMINHO_NORMAL = os.path.join("..", "data", "TEP_FaultFree_Training.RData")
CAMINHO_FALHAS = os.path.join("..", "data", "TEP_Faulty_Training.RData")
FALHAS_PARA_IGNORAR = [3, 9, 15]
USAR_CUDA = True

if not os.path.exists(PASTA_MODELOS): os.makedirs(PASTA_MODELOS)
optuna.logging.set_verbosity(optuna.logging.ERROR)
def destilar_global_fast(df, scaler, hiperplanos):
    if len(df) == 0: return df
    
    # 1. Isola as features
    cols_ignorar = ['faultNumber', 'simulationRun', 'sample', 'target', 'group_id']
    features = df.drop(columns=[c for c in cols_ignorar if c in df.columns], errors='ignore')
    features = features[scaler.feature_names_in_]
    
    # 2. Projeção matricial ultra rápida
    feat_scaled = scaler.transform(features)
    hashes = (np.dot(feat_scaled, hiperplanos) > 0).astype(int)
    
    # 3. BIT PACKING: Transforma 16 bits binários em 1 único número inteiro
    pesos = 2 ** np.arange(hashes.shape[1])
    buckets = np.dot(hashes, pesos)
    
    df_res = df.copy()
    df_res['lsh_bucket'] = buckets
    
    # 4. Mantém um representante por Balde E por Falha (preserva todas as geometrias)
    df_res = df_res.sample(frac=1, random_state=42).drop_duplicates(subset=['lsh_bucket', 'faultNumber'])
    
    return df_res.drop(columns=['lsh_bucket'], errors='ignore')

def split_por_grupo(df, train_frac=0.7, seed=42):
    grupos = df["group_id"].drop_duplicates().to_numpy()
    rng = np.random.RandomState(seed)
    rng.shuffle(grupos)
    corte = int(len(grupos) * train_frac)
    treino_ids = set(grupos[:corte])
    return df[df["group_id"].isin(treino_ids)].copy(), df[~df["group_id"].isin(treino_ids)].copy()

def split_por_grupo(df, train_frac=0.7, seed=42):
    grupos = df["group_id"].drop_duplicates().to_numpy()
    rng = np.random.RandomState(seed)
    rng.shuffle(grupos)
    corte = int(len(grupos) * train_frac)
    treino_ids = set(grupos[:corte])
    return df[df["group_id"].isin(treino_ids)].copy(), df[~df["group_id"].isin(treino_ids)].copy()

# ==============================================================================
# 1. CARREGAMENTO GLOBAL E DESTILAÇÃO ÚNICA (ESTRATÉGIA TURBO)
# ==============================================================================
print("⏳ Carregando bases brutas (.RData)...")
res_n = pyreadr.read_r(CAMINHO_NORMAL)
df_normal_raw = res_n[list(res_n.keys())[0]]
df_normal_raw['faultNumber'] = 0 # Garante que a classe normal seja 0

res_f = pyreadr.read_r(CAMINHO_FALHAS)
df_todas_falhas = res_f[list(res_f.keys())[0]]

# Limpa o transiente inicial das falhas de uma vez
df_todas_falhas = df_todas_falhas[df_todas_falhas['sample'] > 20].copy()

# Cria IDs únicos para agrupamento seguro
df_normal_raw["group_id"] = "normal_f0_run" + df_normal_raw["simulationRun"].astype(str)
df_todas_falhas["group_id"] = "falha_f" + df_todas_falhas["faultNumber"].astype(str) + "_run" + df_todas_falhas["simulationRun"].astype(str)

# Junta a fábrica toda e faz 1 único split de Treino/Teste
df_global = pd.concat([df_todas_falhas, df_normal_raw], ignore_index=True)
df_train_global, df_test_global = split_por_grupo(df_global, train_frac=0.7, seed=42)

print("\n💎 Iniciando Destilação Geométrica Global (LSH com Bit Packing)...")
amostras_brutas_total = len(df_train_global)
cols_ignorar = ['faultNumber', 'simulationRun', 'sample', 'target', 'group_id']
features_treino = df_train_global.drop(columns=[c for c in cols_ignorar if c in df_train_global.columns], errors='ignore')

# Scaler e LSH treinados apenas no bloco de treino (Zero Data Leakage)
scaler_global = StandardScaler().fit(features_treino)
num_ands = 16
hiperplanos_globais = np.random.RandomState(42).randn(features_treino.shape[1], num_ands)

# A mágica acontece aqui:
df_treino_destilado = destilar_global_fast(df_train_global, scaler_global, hiperplanos_globais)
print(f"✅ Planta destilada de {amostras_brutas_total} para {len(df_treino_destilado)} amostras base!\n")

# ==============================================================================
# 2. LOOP DE TREINAMENTO (DESTILAÇÃO TRIPLA E SIMÉTRICA)
# ==============================================================================

tempo_inicio_global = time.time()
estatisticas_gerais = [] # Lista para guardar os dados de todos os especialistas

for f_id in range(1, 21):
    if f_id in FALHAS_PARA_IGNORAR: continue
    
    tempo_inicio_esp = time.time()

    print(f"\n=======================================================")

    print(f"\n=======================================================")
    print(f"🚀 TREINANDO ESPECIALISTA DE ELITE: FALHA {f_id}")
    print(f"=======================================================")
    
    # ==========================================================================
    # 2.1 MONTAGEM DO TREINO (Puxando dos dados já destilados)
    # ==========================================================================
    df_alvo_tr = df_treino_destilado[df_treino_destilado['faultNumber'] == f_id].copy()
    df_alvo_tr['target'] = 1
    
    # Tudo que não for a Falha Alvo (incluindo Normal e outras falhas) vira 0
    df_c0_tr = df_treino_destilado[df_treino_destilado['faultNumber'] != f_id].copy()
    df_c0_tr['target'] = 0

    # Balanceamento Treino
    n_tr = min(len(df_alvo_tr), len(df_c0_tr))
    
    # LINHA CORRIGIDA ABAIXO (Onde estava o erro):
    df_train = pd.concat([
        df_alvo_tr.sample(n=n_tr, random_state=42),
        df_c0_tr.sample(n=n_tr, random_state=42)
    ]).reset_index(drop=True)
    
    amostras_treino_final = len(df_train) 
    percentual_removido = 100 * (1 - (amostras_treino_final / amostras_brutas_total))
    
    
    # ==========================================================================
    # 2.2 PREPARANDO TESTE (Intacto, Sem LSH, puxando do Test Global)
    # ==========================================================================
    df_alvo_ts = df_test_global[df_test_global['faultNumber'] == f_id].copy()
    df_alvo_ts['target'] = 1
    
    df_ts_c0 = df_test_global[df_test_global['faultNumber'] != f_id].copy()
    df_ts_c0['target'] = 0

    # Balanceamento Teste
    df_test = pd.concat([
        df_alvo_ts,
        df_ts_c0.sample(n=min(len(df_alvo_ts), len(df_ts_c0)), random_state=42)
    ]).reset_index(drop=True)

    # 2.5 Variáveis e SHAP
    features_cols = [c for c in df_train.columns if c not in ['faultNumber', 'simulationRun', 'sample', 'target', 'group_id']]
    X_train, y_train, groups_tr = df_train[features_cols], df_train['target'], df_train['group_id']
    X_test, y_test = df_test[features_cols], df_test['target']

    model_base = xgb.XGBClassifier(tree_method='hist', device='cuda' if USAR_CUDA else 'cpu', random_state=42).fit(X_train, y_train)
    shap_v = shap.TreeExplainer(model_base).shap_values(X_train.sample(min(1000, len(X_train)), random_state=42))
    if isinstance(shap_v, list): shap_v = shap_v[1]
    
    top_10 = pd.Series(np.abs(shap_v).mean(0), index=features_cols).nlargest(10).index.tolist()
    print(f"✅ Top 10: {', '.join(top_10)}")

    # 2.6 Optuna (Validação Cruzada por Runs)
    def objective(trial):
        ps = {'n_estimators': trial.suggest_int('n_estimators', 100, 300),
              'max_depth': trial.suggest_int('max_depth', 4, 8),
              'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2)}
        m = xgb.XGBClassifier(**ps, tree_method='hist', device='cuda' if USAR_CUDA else 'cpu', random_state=42)
        return cross_val_score(m, X_train[top_10], y_train, cv=StratifiedGroupKFold(n_splits=3), groups=groups_tr, scoring='accuracy').mean()

    # ... (o começo da 2.6 fica igual)
    
    print("⏳ Otimizando hiperparâmetros no Optuna (Aguarde alguns segundos)...")
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)

    # 2.7 Treino Final e Salvamento
    best_model = xgb.XGBClassifier(**study.best_params, tree_method='hist', device='cuda' if USAR_CUDA else 'cpu').fit(X_train[top_10], y_train)
    y_pred = best_model.predict(X_test[top_10])
    
    # --- CÁLCULO DAS MÉTRICAS ---
    acc_final = accuracy_score(y_test, y_pred)
    prec_final = precision_score(y_test, y_pred, zero_division=0)
    rec_final = recall_score(y_test, y_pred, zero_division=0)
    f1_final = f1_score(y_test, y_pred, zero_division=0)

    # ... (Cálculo do acc_final, prec_final, etc)
    
    # --- ADICIONE ESTE BLOCO AQUI ---
    tempo_fim_esp = time.time()
    tempo_gasto_esp = tempo_fim_esp - tempo_inicio_esp
    
    # --- SALVANDO ESTATÍSTICAS ---
    estatisticas_gerais.append({
        'falha': f_id,
        'amostras_brutas': amostras_brutas_total,
        'amostras_finais': amostras_treino_final,
        'percentual_removido': percentual_removido, # <--- Faltou esse aqui!
        'tempo_segundos': tempo_gasto_esp,
        'acuracia': acc_final,
        'precisao': prec_final,
        'recall': rec_final,
        'f1': f1_final
    })
    # --------------------------------
    
    
    # --- RELATÓRIO VISUAL ---
    print(f"\n📊 RELATÓRIO FINAL DO ESPECIALISTA (FALHA {f_id}):")
    print(f"   ⏱️ Tempo de Treino : {tempo_gasto_esp:.2f} segundos")
    print(f"   📦 Dados Brutos    : {amostras_brutas_total} linhas")
    print(f"   🗜️ Dados com LSH   : {amostras_treino_final} linhas (Redução de {percentual_removido:.2f}%)")
    print(f"   ⚙️ Parâmetros Otimizados: {study.best_params}")
    print(f"   📈 Acurácia Global : {acc_final*100:.2f}%")
    print(f"   🎯 Precisão        : {prec_final*100:.2f}% (Quando apita, a chance de ser real)")
    print(f"   🔎 Recall          : {rec_final*100:.2f}% (Quantas falhas ele conseguiu 'pescar')")
    print(f"   ⚖️ F1-Score        : {f1_final*100:.2f}% (Balanço Geral)")
    
    pacote = {'falha_id': f_id, 'features': top_10, 'modelo': best_model, 'acuracia': acc_final,
              'precisao': prec_final, 'recall': rec_final, 'f1': f1_final}
    joblib.dump(pacote, os.path.join(PASTA_MODELOS, f"especialista_f{f_id}.joblib"))
    print(f"💾 Modelo F{f_id} salvo com sucesso!\n")


# ==============================================================================
# 3. RELATÓRIO GLOBAL DE ENGENHARIA (COM LSH)
# ==============================================================================
tempo_fim_global = time.time()
tempo_total_minutos = (tempo_fim_global - tempo_inicio_global) / 60

# Cálculos das médias
media_amostras_finais = np.mean([e['amostras_finais'] for e in estatisticas_gerais])
media_remocao = np.mean([e['percentual_removido'] for e in estatisticas_gerais])
media_acuracia = np.mean([e['acuracia'] for e in estatisticas_gerais]) * 100
media_precisao = np.mean([e['precisao'] for e in estatisticas_gerais]) * 100
media_recall = np.mean([e['recall'] for e in estatisticas_gerais]) * 100
media_f1 = np.mean([e['f1'] for e in estatisticas_gerais]) * 100

print("\n" + "="*60)
print("🏆 RESUMO GLOBAL DO PIPELINE COM LSH (TURBO)")
print("="*60)
print(f"⏱️ Tempo Total de Execução: {tempo_total_minutos:.2f} minutos")
print(f"📊 Média de Amostras Treinadas por Especialista: {media_amostras_finais:.0f} linhas")
print(f"✂️ Eficiência do LSH (Média de Compressão): {media_remocao:.2f}% dos dados descartados")
print("-" * 60)
print(f"📈 Média de Acurácia Global : {media_acuracia:.2f}%")
print(f"🎯 Média de Precisão        : {media_precisao:.2f}%")
print(f"🔎 Média de Recall          : {media_recall:.2f}%")
print(f"⚖️ Média de F1-Score        : {media_f1:.2f}%")
print("="*60)
